In [1]:
import osmnx as ox
import networkx as nx
import pandas as pd
from pathlib import Path

In [2]:
class ServiceNotAvailableError(Exception):
    "Excepción que indica que la navegación no está disponible en este momento"
    pass


class AddressNotFoundError(Exception):
    "Excepción que indica que una dirección buscada no existe en la base de datos"
    pass

In [25]:
import ast
import math
import re
from collections import Counter
from pathlib import Path
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import networkx as nx
import osmnx as ox
import pandas as pd

STREET_FILE_NAME="direcciones.csv"

PLACE_NAME = "Madrid, Spain"
MAP_FILE_NAME="madrid.graphml"

MAX_SPEEDS={'living_street': '20',
'residential': '30',
'primary_link': '40',
'unclassified': '40',
'secondary_link': '40',
'trunk_link': '40',
'secondary': '50',
'tertiary': '50',
'primary': '50',
'trunk': '50',
'tertiary_link':'50',
'busway': '50',
'motorway_link': '70',
'motorway': '100'}


class ServiceNotAvailableError(Exception):
    "Excepción que indica que la navegación no está disponible en este momento"
    pass


class AddressNotFoundError(Exception):
    "Excepción que indica que una dirección buscada no existe en la base de datos"
    pass


############## Parte 2 ##############

def comprobar_fichero():
    
    origen_path = Path("direcciones.csv")
    temporal_path = Path("direcciones_2_utf8.csv")

    try:
        # Comprueba si ya es UTF-8 válido
        with origen_path.open("r", encoding="utf-8", newline="") as f:
            pass

    except UnicodeDecodeError:
        # Si no era UTF-8 (p. ej., cp1252), lo convertimos y reemplazamos
        with origen_path.open("r", encoding="cp1252", newline="") as origen:
            contenido = origen.read()

        with temporal_path.open("w", encoding="utf-8", newline="") as destino:
            destino.write(contenido)

        origen_path.unlink()
        temporal_path.rename(origen_path)
        print("Convertido de cp1252 a UTF-8 y reemplazado direcciones.csv.")


def carga_callejero() -> pd.DataFrame:
    """ Función que carga el callejero de Madrid, lo procesa y devuelve
    un DataFrame con los datos procesados
    
    Args: None
    Returns:
        DataFrame: dataframe con los datos del callejero procesados.
    Raises:
        FileNotFoundError si el fichero csv con las direcciones no existe
    """

    try:
        columnas_str = [
            'Clase de la via',
            'Nombre de la vía',
            'Literal de numeracion',
            'Partícula de la vía'
        ]

        columnas_float = [
            'Latitud en S R  ETRS89 WGS84',
            'Longitud en S R  ETRS89 WGS84'
        ]

        columnas = columnas_str + columnas_float

        raw = pd.read_csv('direcciones.csv', delimiter=';', usecols= columnas, skipinitialspace=True)
        callejero = raw.copy()

        callejero.rename(columns={
            'Clase de la via':'clase',
            'Nombre de la vía':'nombre',
            'Literal de numeracion':'numero',
            'Latitud en S R  ETRS89 WGS84':'latitud',
            'Longitud en S R  ETRS89 WGS84':'longitud',
            'Partícula de la vía':'particula'
        }, inplace=True)

        for col in ('clase', 'particula', 'nombre', 'numero'):
            callejero[col] = callejero[col].fillna('').astype(str).str.strip().str.upper()

        callejero['latitud'] = callejero['latitud'].map(dms_a_decimal)
        callejero['longitud'] = callejero['longitud'].map(dms_a_decimal)

        return callejero



    except FileNotFoundError:
        print('--- ERROR ---\n\t - El fichero de callejero no existe')

def dms_a_decimal(valor: str) -> float:
    texto = valor.strip().upper()
    signo = -1.0 if ("S" in texto or "W" in texto or "O" in texto) else 1.0

    for sep in ("º", "°", "'", '"', "N", "S", "E", "W", "O"):
        texto = texto.replace(sep, " ")

    partes = [float(p) for p in texto.split() if p]
    grados = partes[0] if len(partes) > 0 else 0.0
    minutos = partes[1] if len(partes) > 1 else 0.0
    segundos = partes[2] if len(partes) > 2 else 0.0

    return signo * (grados + minutos / 60.0 + segundos / 3600.0)

def busca_direccion(direccion: str, callejero: pd.DataFrame) -> tuple[float, float]:
    """
    Función que busca una dirección, dada en el formato
        calle, numero
    en el DataFrame callejero de Madrid y devuelve el par (latitud, longitud) en grados de la
    ubicación geográfica de dicha dirección.

    Args:
        direccion (str): Nombre completo de la calle con número, en formato "Calle, num"
        callejero (DataFrame): DataFrame con la información de las calles

    Returns:
        Tuple[float, float]: Par de float (latitud, longitud) de la dirección buscada, expresados en grados

    Raises:
        AddressNotFoundError: Si la dirección no existe en la base de datos

    Example:
        busca_direccion("Calle de Alberto Aguilera, 23", data) = (40.42998055555555, -3.7112583333333333)
        busca_direccion("Calle de Alberto Aguilera, 25", data) = (40.43013055555555, -3.7126916666666667)
    """
    direccion_limpia = direccion.strip().upper().replace(',', '')
    direccion_lista = [tok for tok in direccion_limpia.split(' ') if tok]

    if len(direccion_lista) < 3:
        raise AddressNotFoundError('La dirección pasada no tiene una estructura válida para buscar')

    clase = direccion_lista[0]  # toda clase es de longitud uno

    available_particles = ('DE', 'DEL', 'DE LA', 'DE LOS', 'DE LAS', 'A LA', 'AL', 'LA', 'LOS', 'LAS')

    cand_1 = direccion_lista[1] if direccion_lista[1] in available_particles else ''
    cand_2 = direccion_lista[2] if (cand_1 and len(direccion_lista) > 2) else ''

    particula, i = determinar_particula(cand_1, cand_2, available_particles)

    nombre, numero = determinar_nombre(direccion_lista[i:])
    if not nombre or not numero.isdecimal():
        raise AddressNotFoundError(f'No se encontró la dirección: {direccion_limpia}')

    literal_buscado = f'NUM{int(numero):06d}'  # genera el número usando el prefijo y rellenando con ceros

    candidatas = callejero[
        (callejero['clase'] == clase)
        & (callejero['nombre'] == nombre)
        & (callejero['particula'] == particula)
        & (callejero['numero'] == literal_buscado)
    ]

    if candidatas.empty:
        raise AddressNotFoundError(f'No se encontró la dirección: {direccion_limpia}')

    lat = float(candidatas.iloc[0]['latitud'])
    lon = float(candidatas.iloc[0]['longitud'])

    return (lat, lon)

def determinar_particula(candidato_1: str, candidato_2: str, posibilidades: tuple[str, ...]) -> tuple[str, int]:
    """
    Función que determina la partícula de la calle pasada por argumento
    a la función buscar dirección.

    Args:
        candidato_1: str
        candidato_2: str
        posibilidades: tuple[str, ...]

    Returns:
        tuple[str, int]: se devuelve la partícula y el índice siguiente (2 o 3)

    Raises: None
    """
    concat = candidato_1 + ' ' + candidato_2  # se concatena la otra posible combinación de candidatos
    concat = concat.strip()  # Evitamos espacios al final en caso de que candidato 2 venga vacío

    # Si la concatenación es una posibilidad, quiere decir que es la partícula de longitud 2
    if concat and concat in posibilidades:
        return (concat, 3)

    # Si no, miramos si el candidato uno es una posibilidad
    elif candidato_1 and candidato_1 in posibilidades:
        return (candidato_1, 2)

    # Si no, es que no tenemos partícula
    return ('', 1)


def determinar_nombre(direccion_lista: list[str]) -> tuple[str, str]:
    """
    Función que devuelve el nombre de la calle y el número de la dirección pasada.

    Args:
        direccion_lista: list[str]

    Returns:
        tuple[str, str]: nombre de la calle pasada por argumento y número

    Raises: None
    """
    if not direccion_lista:
        return ('', '')

    # Se establece el límite máximo de indexación para el nombre
    i, limite = 0, len(direccion_lista) - 1
    nombre = ''

    # Mientras que el elemento no sea un número, se añade al nombre
    while not direccion_lista[i].isdecimal():
        nombre += direccion_lista[i] + ' '

        # Si al indexar el siguiente valor se sale de la lista, para el bucle ---> no hay número en la dirección
        if i + 1 > limite:
            break
        else:
            i += 1

    if direccion_lista[i].isdecimal():
        number = direccion_lista[i]
    else:
        number = ''

    # Devuelve el nombre sin espacio final y el número encontrado (o cadena vacía)
    return (nombre.strip(), number)


datos = busca_direccion('Calle Gran via, 1', callejero)
print(datos)

(40.41914722222222, -3.698066666666667)


In [4]:
callejero = carga_callejero()
display(callejero)

,clase,particula,nombre,numero,longitud,latitud
0,CALLE,DEL,26 DE ABRIL,NUM000002,-3.601525,40.437583
1,AUTOVIA,,A-1,KM.001000EN,-3.673222,40.489394
2,AUTOVIA,,A-1,KM.001000SA,-3.672431,40.489292
3,AUTOVIA,,A-1,KM.001100EN,-3.665992,40.496697
4,AUTOVIA,,A-1,KM.001100SA,-3.665242,40.496189
...,...,...,...,...,...,...
215600,PASEO,DEL,ZURRON,NUM000049,-3.584978,40.469658
215601,PASEO,DEL,ZURRON,NUM000051,-3.584258,40.469661
215602,PASEO,DEL,ZURRON,NUM000053,-3.583919,40.469817
215603,PASEO,DEL,ZURRON,NUM000055,-3.583622,40.469772


In [11]:
callejero[callejero['nombre'] == 'GRAN VIA']

,clase,particula,nombre,numero,longitud,latitud
90437,CALLE,,GRAN VIA,NUM000001,-3.698067,40.419147
90438,CALLE,,GRAN VIA,NUM000002,-3.697394,40.419225
90439,CALLE,,GRAN VIA,NUM000003,-3.698456,40.419253
90440,CALLE,,GRAN VIA,NUM000004,-3.697642,40.419294
90441,CALLE,,GRAN VIA,NUM000005,-3.698622,40.419306
...,...,...,...,...,...,...
90516,CALLE,,GRAN VIA,NUM000073,-3.710778,40.423117
90517,CALLE,,GRAN VIA,NUM000074,-3.709881,40.422986
90518,CALLE,,GRAN VIA,NUM000076,-3.710225,40.423208
90519,CALLE,,GRAN VIA,NUM000078,-3.710272,40.423236


In [21]:
datos = busca_direccion('Calle Gran via', callejero)
print(datos)

AddressNotFoundError: No se encontró la dirección: CALLE GRAN VIA